#Extract data from busses API

## Imports

In [0]:
import json
import requests
from pyspark.sql.functions import current_timestamp, lit

# Extracting data from API

In [0]:
BRONZE_TABLE_NAME = "busses_stream.ingestion.ingestion_busses"

API_URL = "https://przystanki.bialystok.pl/portal/getRunningVehicles.json" 

try:
    response = requests.get(API_URL, timeout=30)
    response.raise_for_status()
    payload = response.json()
except Exception as e:
    print(f"API fetch skipped or failed ({e}). Extracting from raw payload variable...")
    payload = {
        "vehicles": [
        ]
    }

vehicles_list = payload.get("vehicles", [])

if not vehicles_list:
    print("Warning: No vehicle records found in the payload.")

df_vehicles = spark.createDataFrame(vehicles_list)
df_bronze = (
    df_vehicles
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_system", lit("Białystok Transit API"))
)

(
    df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE_NAME)
)

print(f"Successfully ingested {df_bronze.count()} records into {BRONZE_TABLE_NAME}.")